In [171]:
from langchain.agents import create_agent
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace

In [172]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage,ToolMessage

In [173]:
load_dotenv()

True

## Database

In [174]:
tasks = []

## Tools Creation

In [175]:
from langchain.tools import tool

In [176]:
@tool
def add_task(task:str , priority: str):
    """This tool is used to add the task"""
    task_id = len(tasks) + 1
    tasks.append({
        "task_id" : task_id,
        "task" : task,
        "Priority" : priority,
        "Completed" : False
    })

    return f"Task added Successfully. Current tasks are - {tasks}"

@tool
def list_tasks():
    """This tool was used to return all the tasks"""

    return tasks

@tool
def Complete_Task(id:int):
    """Complete a task using its task_id.
    Before calling this tool, use list_tasks if the user
    identifies a task by description or some other parameters rather than ID """
    task_idx = None
    for idx,task in enumerate(tasks):
        if task["task_id"] == id:
            task_idx = idx
            break

    if task_idx is None:
        return f"There is no such task available. Current tasks are - {tasks}"
    
    tasks[task_idx]["Completed"] = True
    return f"Updated the Tasks. Current Tasks are - {tasks}"

@tool
def Delete_Task(id:int):
    """Delete the task using the task_id
    Before calling this tool, use list_tasks if the user
    identifies a task by description or some other parameters rather than ID """
    task_idx = None
    for idx,task in enumerate(tasks):
        if task["task_id"] == id:
            task_idx = idx
            break
    if task_idx is None:
        return f"There is no such task available. Current tasks are - {tasks}"

    tasks.pop(task_idx)

    return f"Task Deleted successfully. currently available tasks are - {tasks}"


In [177]:
tools = {
    tool.name : tool for tool in [add_task,list_tasks,Complete_Task,Delete_Task]
}

## middlewares

In [178]:
from langchain.agents.middleware import (
    wrap_tool_call,
    ModelRequest,
    AgentState,
    before_model
)

In [208]:


@wrap_tool_call
def ToolLoggerMiddleware(request,handler):
    print("request : ---", request.tool_call.get("name"),"\n")
    
    if request.tool_call.get("name") == "Delete_Task":
            res = input("Do you really want to delete the task (Y) or (N) : ")
            if res.upper() == "N":
                return ToolMessage(
                    content="The user cancelled the deletion. Do not delete the task.",
                    tool_call_id=request.tool_call.get("id")
                )

    print("\nhandler : ", handler)

    result = handler(request)

    print("\nresult :",result)
    
    return result

In [180]:
class ModelCounter(AgentState):
    """ Used for gaining number of model calls"""
    model_call_count : int = 0

In [181]:
@before_model(state_schema=ModelCounter)
def CounterMiddleWare(state,runtime):
    # print("\nCounter middleware request : ",request)
    print("\nmodel_call_count : ",state.get("model_call_count",0)+1)
    return {
        "model_call_count" : state.get("model_call_count",0) + 1
    }


## Agent Creation

In [182]:
import os

In [183]:
llm = HuggingFaceEndpoint(
    repo_id=os.getenv("HF_MODEL"),
    max_new_tokens= 1024,
    provider="auto",
    temperature=0.7,
    huggingfacehub_api_token=os.getenv("HF_TOKEN"),
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

agent = create_agent(
    model=model,
    tools=[add_task,list_tasks,Complete_Task,Delete_Task],
    system_prompt="You are an AI Agent which was used to manage the tasks.",
    middleware=[ToolLoggerMiddleware,CounterMiddleWare]
)

In [211]:
messages = []

query = input("Ask your query: ")

messages.append(HumanMessage(query))

response = agent.invoke({
    "messages" : messages
})

messages.append(response)

print("\nmodel calls : ",response["model_call_count"])

# print("Model Response: \n",response)



model_call_count :  1
request : --- list_tasks 


handler :  <function ToolNode._run_one.<locals>.execute at 0x000001F0E345B100>

result : content='[{"task_id": 1, "task": "Doing project", "Priority": "high", "Completed": false}, {"task_id": 2, "task": "Taking shower", "Priority": "low", "Completed": false}, {"task_id": 3, "task": "Watching movie", "Priority": "medium", "Completed": false}]' name='list_tasks' tool_call_id='chatcmpl-tool-a34017f91ac871ac'

model_call_count :  2
request : --- Delete_Task 


handler :  <function ToolNode._run_one.<locals>.execute at 0x000001F0E345B060>

result : content="Task Deleted successfully. currently available tasks are - [{'task_id': 1, 'task': 'Doing project', 'Priority': 'high', 'Completed': False}, {'task_id': 3, 'task': 'Watching movie', 'Priority': 'medium', 'Completed': False}]" name='Delete_Task' tool_call_id='chatcmpl-tool-9a5067936f83744d'

model_call_count :  3

model calls :  3


In [212]:
print(response["messages"][-1].content)



Deleted the "Taking shower" task. Your remaining tasks are:

1. **Doing project** (high priority)
3. **Watching movie** (medium priority)


In [213]:
print(tasks)

[{'task_id': 1, 'task': 'Doing project', 'Priority': 'high', 'Completed': False}, {'task_id': 3, 'task': 'Watching movie', 'Priority': 'medium', 'Completed': False}]


In [187]:
print(messages)

[HumanMessage(content='delete the lowe priority task', additional_kwargs={}, response_metadata={}, id='266959e0-3182-4092-bb5c-6a6230e5b7ff'), {'messages': [HumanMessage(content='delete the lowe priority task', additional_kwargs={}, response_metadata={}, id='266959e0-3182-4092-bb5c-6a6230e5b7ff'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'list_tasks', 'description': None}, 'id': 'chatcmpl-tool-9d7a2e00b5dd7248', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 554, 'total_tokens': 598}, 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10826-17fd-77e0-89ef-0fe995c917bb-0', tool_calls=[{'name': 'list_tasks', 'args': {}, 'id': 'chatcmpl-tool-9d7a2e00b5dd7248', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 554, 'output_tokens': 44, 'total_tokens': 598}), ToolMessage(content=[]

In [188]:
for message in response["messages"][-1]:
    print(message)
    print("-" * 100)


('content', "\n\nThere are currently no tasks in your task list, so there's nothing to delete. Would you like me to add some tasks instead?")
----------------------------------------------------------------------------------------------------
('additional_kwargs', {})
----------------------------------------------------------------------------------------------------
('response_metadata', {'token_usage': {'completion_tokens': 63, 'prompt_tokens': 584, 'total_tokens': 647}, 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None})
----------------------------------------------------------------------------------------------------
('type', 'ai')
----------------------------------------------------------------------------------------------------
('name', None)
----------------------------------------------------------------------------------------------------
('id', 'lc_run--01a10826-1ef2-7c50-b37d-502043464646-0')
--------------------------

In [203]:
print(response)

{'messages': [HumanMessage(content='delete the task of taking shower', additional_kwargs={}, response_metadata={}, id='041e345d-b2b4-4da5-b0d6-e66bf4114078'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{}', 'name': 'list_tasks', 'description': None}, 'id': 'chatcmpl-tool-9eeadf1f28215087', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 554, 'total_tokens': 599}, 'model_name': 'Qwen/Qwen3.8-27B', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1082a-47c2-7773-906f-80b05978298b-0', tool_calls=[{'name': 'list_tasks', 'args': {}, 'id': 'chatcmpl-tool-9eeadf1f28215087', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 554, 'output_tokens': 45, 'total_tokens': 599}), ToolMessage(content='[{"task_id": 1, "task": "Doing project", "Priority": "high", "Completed": false}, {"task_id": 2, "task": "Taking shower", "Priority": "low", 